# 03. E02: item-item по совместному просмотру

Стадия: retrieval. Код модели: `src/item_knn.py`, оценка: `src/evaluate.py`. Протокол и критерий успеха: `docs/system_design.md` (`ndcg@10` не ниже 90% от популярности и `coverage@10` в 10 раз больше). Запись для журнала: `experiments/README.md`, E02.

Идея: если многие, кто досмотрел A, досмотрели и B, то A и B похожи. Для пользователя балл кандидата — сумма его сходств с аниме из `input`-истории. Вклад пользователя в сходство взвешен (`form`, по умолчанию `1/n`), сходство нормировано на размеры аудиторий (косинус).

## Загрузка и построение модели

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))  # код проекта лежит в src/
import evaluate as ev
import item_knn as knn

table, catalog = ev.load_interactions()
cases = ev.build_cases(table, "val")

matrix, train_users, item_ids = knn.build_train_matrix(table, catalog)  # (пользователи train, аниме каталога)
histories = knn.build_histories(table, "val", item_ids)  # что val-пользователь уже досмотрел (input)

matrix.shape, len(cases)

## Запуск с весом `1/n` и сравнение с популярностью

`user_weights(matrix, form=0)` — вес `1/n`; `item_similarity` строит матрицу сходства; `make_item_knn_ranker` превращает её в ранжер. Таблица и проверка критерия.

In [ ]:
weights = knn.user_weights(matrix, form=0)
similarity = knn.item_similarity(matrix, weights)
knn.check_similarity(similarity)

rankers = {
    "popularity": ev.make_popularity_ranker(ev.popularity_scores(table)),
    "item_knn": knn.make_item_knn_ranker(similarity, item_ids, histories),
}
results = pd.DataFrame({name: ev.evaluate(r, cases, catalog) for name, r in rankers.items()})
print(knn.check_criterion(results, "item_knn"))
results.round(4)

Как читать критерий: `accuracy` — `ndcg@10` не ниже 90% от популярности, `coverage` — `coverage@10` в 10 раз больше, чем у популярности.

## Подбор формы веса пользователя

Три формы веса: `0` — `1/n`, `1` — `1/sqrt(n)`, `2` — `1/log(1+n)`. Всё остальное одинаково. Это подбор параметра на val, а не новая идея: сравниваем точность и покрытие.

In [ ]:
rows = {}
for form in (0, 1, 2):
    sim = knn.item_similarity(matrix, knn.user_weights(matrix, form=form))
    ranker = knn.make_item_knn_ranker(sim, item_ids, histories)
    rows[f"form={form}"] = ev.evaluate(ranker, cases, catalog)

forms = pd.DataFrame(rows)
forms.loc[["hit_rate@10", "recall@10", "ndcg@10", "coverage@10"]].round(4)

## Проверка здравым смыслом: на что похоже аниме

Для выбранного аниме выводим 10 самых похожих по матрице сходства. Если модель работает, рядом должны оказаться близкие по жанру, студии и сюжету тайтлы.

In [ ]:
names = pd.read_csv(ev.INTERACTIONS.parent / "shikimori_v1" / "anime.csv", usecols=["anime_id", "name", "russian"]).set_index("anime_id")


def similar_to(anime_id: int, n: int = 10) -> pd.DataFrame:
    """Топ-n самых похожих аниме по матрице сходства."""
    column = int(np.searchsorted(item_ids, anime_id))
    top = np.argsort(-similarity[column])[:n]
    ids = item_ids[top]
    return pd.DataFrame({"anime_id": ids, "name": names.loc[ids, "name"].values, "similarity": similarity[column, top].round(3)})


similar_to(1)  # 1 = Cowboy Bebop; поменяй на любой anime_id из каталога

## Лента глазами для одного val-пользователя

Что человек уже досмотрел (из `input`), что модель рекомендует ему в топ-10 и какие из рекомендаций совпали со спрятанными ответами (`target`).

In [ ]:
user_id = list(cases)[0]  # поменяй индекс, чтобы посмотреть другого пользователя
seen, targets = cases[user_id]
ranker = rankers["item_knn"]

watched = item_ids[histories[user_id]]
print("Досмотрел (первые 15):")
print(names.loc[watched[:15], "name"].to_string())

candidates = catalog[~np.isin(catalog, list(seen))]
top = ranker(user_id, candidates)[:10]
pd.DataFrame({"anime_id": top, "name": names.loc[top, "name"].values, "угадано": [a in targets for a in top]})

Вывод: _(что показали числа и сама лента: стала ли модель точнее популярности; что с покрытием и почему оно не дотягивает до критерия; какая форма веса лучше; выглядят ли рекомендации осмысленно, повторяются ли у разных пользователей одни и те же аниме)_